
## MDP
MDP, or Markov Decision Process : is the main formalism used in Reinforcement Learning


**fully observable** environment, the agent state is the same as the environment state -> formally a markov decision process

**partially observable** environment the agent state must be somehow computed from the history of observations, since the observations are now incomplete representations of the environment state.
 - Ex: poker playing agent only observe visible cardes, Atari game player has no access to internal state of atari console and program
 - $S_t^a \neq S_t^e$ -> Partially observable MDP -> agent must construct its own state representation (its **belief**)

can be converted to MDP and so behave largely the same way


----

**Finite MDP**: when there are finite states, finite rewards, and finite actions -> apply classic **tabular methods** (Q-tables, Value tables)

**Infinite/Continuous MDP**: 
- continuous state spaces: like robot control (joint angles, velocities), autonomous vehicules (sensor reading, positions) -> **function approximation** (neural networks, linear models)
- continuous action spaces:  like motor torques, steering angles,... -> policy gradient methods, actor-critics
- infinite horizon: no fixed episode length. Use discounting $\gamma$ to ensure finite returns




## Type of tasks

### Episodic tasks

The interaction between an agent and the environment can include a set of independent episodes. In this scenario, every episode starts independently from others and its beginning state is sampled from the distribution of states.

Guaranteed to terminate

Episodes aka **trials**

### Continuing tasks

Some tasks do not have a terminal state. Not always possible to define a cumulative return due to infinite timestamps.

Go on indefinitely


Starting with:
Markov Chains >> Markov Reward Processes >>  Markov Decision Processes


## Markov chains


<img src="../image-9.png" width="500px" />


We don’t need to know the full history of states to know what will happen next, just the current one

State transition matrix, probability of transition from a state (row) to another one (col):

<img src="../image-10.png" width="500px" />


A Markov Chain has **no rewards or actions yet**, it simply defines a random process that generates a set of states, each with the Markov property

<img src="../image-11.png" width="500px" />


Student MDP:

<img src="../image-12.png" width="500px" />


One can sample random set of states. Ex: ```['C1', 'C2', 'C3', 'Pub', 'C1', 'C2', 'Sleep']````

Agent make no decision, only at mercy of chance.  

In [1]:
import numpy as np
from icecream import ic

In [2]:
states = ["Class1", "Class2", "Class3", "Facebook", "Pub", "Pass", "Sleep"]
terminal_states = ["Sleep"]  # , "Pass"]

transitions = {
    "Class1": [
        (0.5, "Class2"),  # Study
        (0.5, "Facebook"),  # Facebook
    ],
    "Class2": [
        (0.8, "Class3"),  # Study
        (0.2, "Sleep"),  # Sleep
    ],
    "Class3": [
        (0.6, "Pass"),
        (0.4, "Pub"),
    ],
    "Facebook": [
        (0.9, "Facebook"),  # Keep browsing
        (0.1, "Class1"),  # Quit
    ],
    "Pub": [
        (0.2, "Class1"),  # Leave pub, go to C1 (0.5 * 0.2)
        (0.4, "Class2"),  # Leave pub, go to C2 (0.5 * 0.4)
        (0.4, "Class3"),  # Leave pub, go to C3 (0.5 * 0.4)
    ],
    "Pass": [(1.0, "Sleep")],  # Terminal state
    "Sleep": [],  # Terminal state
}


transitions_with_terminal_states = {s: [] for s in states}
for s, trans in transitions.items():
    transitions_with_terminal_states[s] = [(p, s_next) for p, s_next in trans]

for t in terminal_states:
    transitions_with_terminal_states[t] = [(0.0, t)]  # we stay at terminal states


transitions_with_terminal_states


{'Class1': [(0.5, 'Class2'), (0.5, 'Facebook')],
 'Class2': [(0.8, 'Class3'), (0.2, 'Sleep')],
 'Class3': [(0.6, 'Pass'), (0.4, 'Pub')],
 'Facebook': [(0.9, 'Facebook'), (0.1, 'Class1')],
 'Pub': [(0.2, 'Class1'), (0.4, 'Class2'), (0.4, 'Class3')],
 'Pass': [(1.0, 'Sleep')],
 'Sleep': [(0.0, 'Sleep')]}

In [3]:
# Create transition matrix from P dictionary
def create_transition_matrix(P, states):
    n = len(states)
    T = np.zeros((n, n))

    # Create state to index mapping
    state_to_idx = {s: i for i, s in enumerate(states)}

    # Fill the matrix
    for state, transitions in P.items():
        i = state_to_idx[state]
        for prob, next_state in transitions:
            j = state_to_idx[next_state]
            T[i, j] = prob

    return T


# Create the transition matrix
P = create_transition_matrix(transitions_with_terminal_states, states)

print("Transition Matrix (rows=from state, cols=to state):")
print("\nStates:", states)
print("\nMatrix:")
print(P)
print("\n" + "=" * 60)

# Verify it's stochastic (rows sum to 1)
row_sums = P.sum(axis=1)
print("\nRow sums (should all be 1.0):")
for i, s in enumerate(states):
    print(f"  {s:>10}: {row_sums[i]:.6f}")


Transition Matrix (rows=from state, cols=to state):

States: ['Class1', 'Class2', 'Class3', 'Facebook', 'Pub', 'Pass', 'Sleep']

Matrix:
[[0.  0.5 0.  0.5 0.  0.  0. ]
 [0.  0.  0.8 0.  0.  0.  0.2]
 [0.  0.  0.  0.  0.4 0.6 0. ]
 [0.1 0.  0.  0.9 0.  0.  0. ]
 [0.2 0.4 0.4 0.  0.  0.  0. ]
 [0.  0.  0.  0.  0.  0.  1. ]
 [0.  0.  0.  0.  0.  0.  0. ]]


Row sums (should all be 1.0):
      Class1: 1.000000
      Class2: 1.000000
      Class3: 1.000000
    Facebook: 1.000000
         Pub: 1.000000
        Pass: 1.000000
       Sleep: 0.000000


## Markov Reward Processes
A Markov reward process is exactly like a Markov chain, except each step taken generates a reward.

<img src="../image-13.png" width="500px" />
<div/>

<img src="../image-14.png" width="500px" />


trajectories we sample will have an associated return value.

<img src="../image-15.png" width="500px" />


with $\gamma=0$, value is same as reward:

<img src="../image-76.jpg" width="500px" />

Gamma (0..1) determines how important future rewards are to the agent. if close to 0, consider immediate reward only.

Gamma=1 (for the rest of course for convenience)

Value function: expectation over future rewards 


<img src="../image-16.png" width="500px" />



Thanks to the law of large numbers, a rough estimate of the true value function for a state can be achieved by sampling many trajectories starting from that state, and averaging the sampled returns




#### Bellman equation


The Bellman equation states a profound idea: 

To evaluate a decision, you don’t need to know the entire future, only what happens next and how good the future looks from there.

$$V(s)={E}[r + \gamma V(s')]$$

value of a state is decomposed in 2 parts:
- the expected immediate reward 
- the discounted value of the next state. 

Approach:
- Instead of planning everything at once, the agent breaks the problem into smaller pieces, evaluating one step at a time.
- Works because of the Markov property, future depends only on the current state and action, not on the entire history that led there
- Finally, repeated Bellman updates converge to a stable solution







<img src="../image-17.png" width="500px" />



as gamma is constant:


<img src="../image-18.png" width="500px" />
<div/>

<img src="../image-19.png" width="500px" />
<div/>

<img src="../image-20.png" width="500px" />


So : **value of a state s is the expectation over the immediate reward plus the discounted expected future reward***


##### How do we get that expected reward of the next state v(St+1)?



<img src="../image-21.png" width="500px" />


= **sum of all state-values weighted by their probability of occurring**. 

Many of the values in this sum will be zero, because you can only transition to some states at each step

Notice that the immediate reward is no longer expressed as an expectation because it’s a constant, and the expectation of a constant is just a constant, so we can pull it out of the expectation.

**Bellman equation** : expresses the state-value in terms of itself .

<img src="../image-22.png" width="500px" />

Nodes b1 and b2 are possible next states reachable from node a



we’re taking the immediate reward r and then averaging over the values of all possible successor states

Self check with former value calculated iteratively:

<img src="../image-23.png" width="500px" />

Different way to solve it:

##### Analytical solution


<img src="../image-24.png" width="500px" />
<div/>

<img src="../image-25.png" width="500px" />

<div/>
<img src="../image-26.png" width="500px" />


Often to large to be used:
- Computational complexity is $O(n^3)$ for n states
- many iteratives methods for large MRP:
    - dynamic programming
    - monte-carlo evaluation
    - temporal-difference learning





In [16]:
## Analytical Value Calculation with bellman Equations

# Rewards
# ['Class1', 'Class2', 'Class3', 'Facebook', 'Pub', 'Pass', 'Sleep']
_rewards = [-2, -2, -2, -1, 1, 10, 0]

gamma = 1
R = np.array(_rewards)
P_ = np.matrix(P)
ic(P_)
I = np.identity(P_.shape[0])
ic(I)
analytical_solution = np.dot(np.linalg.inv((I - gamma * P_)), R)
analytical_solution = analytical_solution.tolist()[0]


for state in range(len(states)):
    print(states[state], analytical_solution[state])



Class1 -12.543209876543207
Class2 1.4567901234567913
Class3 4.320987654320987
Facebook -22.54320987654321
Pub 0.8024691358024687
Pass 10.0
Sleep 0.0


ic| P_: matrix([[0. , 0.5, 0. , 0.5, 0. , 0. , 0. ],
                [0. , 0. , 0.8, 0. , 0. , 0. , 0.2],
                [0. , 0. , 0. , 0. , 0.4, 0.6, 0. ],
                [0.1, 0. , 0. , 0.9, 0. , 0. , 0. ],
                [0.2, 0.4, 0.4, 0. , 0. , 0. , 0. ],
                [0. , 0. , 0. , 0. , 0. , 0. , 1. ],
                [0. , 0. , 0. , 0. , 0. , 0. , 0. ]])
ic| I: array([[1., 0., 0., 0., 0., 0., 0.],
              [0., 1., 0., 0., 0., 0., 0.],
              [0., 0., 1., 0., 0., 0., 0.],
              [0., 0., 0., 1., 0., 0., 0.],
              [0., 0., 0., 0., 1., 0., 0.],
              [0., 0., 0., 0., 0., 1., 0.],
              [0., 0., 0., 0., 0., 0., 1.]])


#### Iterative solution (using bellman)

In [17]:
r = {
    "Class1": -2.0,
    "Class2": -2.0,
    "Class3": -2.0,
    "Facebook": -1.0,
    "Pub": 1.0,
    "Sleep": 0.0,
    "Pass": 10.0,
}
r

{'Class1': -2.0,
 'Class2': -2.0,
 'Class3': -2.0,
 'Facebook': -1.0,
 'Pub': 1.0,
 'Sleep': 0.0,
 'Pass': 10.0}

In [27]:
possible_transitions = transitions["Class1"]
ic(possible_transitions)
#get_transitions("Sleep")
probs = [t[0] for t in possible_transitions]
probs

ic| possible_transitions: [(0.5, 'Class2'), (0.5, 'Facebook')]


[0.5, 0.5]

In [29]:
def get_transitions(state):
    """Transition function for the MDP"""
    return transitions_with_terminal_states[state]


def reward(state) -> float:
    """Reward function for the MDP"""
    return float(r[state])


def step(state):
    """Take a step in the Markov Chain from current state"""
    if state in terminal_states:
        return state

    if state not in transitions:
        raise ValueError(f"Invalid state: {state}")

    # Get possible transitions
    possible_transitions = transitions[state]

    if not possible_transitions:
        return state

    # Sample next state based on probabilities
    probs = [t[0] for t in possible_transitions]
    idx = np.random.choice(len(possible_transitions), p=probs)
    _, next_state = possible_transitions[idx]

    return next_state


print("from Class1:", step("Class1"))
print("from Class3:", step("Class3"))
get_transitions("Class1")


from Class1: Facebook
from Class3: Pass


[(0.5, 'Class2'), (0.5, 'Facebook')]

In [30]:
gamma = 1.0
theta = 1e-12
max_iters = 30_000

V = {s: 0.0 for s in states}

# Terminal states yield their reward once and then the episode ends.
# So we set their values directly, and do not update them in the loop.
for t in terminal_states:
    V[t] = reward(t)
    

print("Initial Values:")
for s in states:
    print(f"  {s:>10}: {V[s]:.6f}")

print(f"\nRunning Value Iteration... {max_iters} (prints every 1000 iterations)")
for i in range(0, max_iters):
    delta = 0.0

    if i % 1000 == 0:
        print(i, V)

    for s in states:
        if s in terminal_states:
            continue
        v_old = V[s]
        exp_next = sum(p * V[s_next] for p, s_next in get_transitions(s))
        V[s] = reward(s) + gamma * exp_next
        delta = max(delta, abs(v_old - V[s]))
    if delta < theta:
        print(
            f"Converged after {i} iterations with delta={delta:.6e} < theta={theta:.6e}"
        )
        break


print("\nFinal Values after Value Iteration:")
for s in states:
    print(f"  {s:>10}: {V[s]:.6f}")

Initial Values:
      Class1: 0.000000
      Class2: 0.000000
      Class3: 0.000000
    Facebook: 0.000000
         Pub: 0.000000
        Pass: 0.000000
       Sleep: 0.000000

Running Value Iteration... 30000 (prints every 1000 iterations)
0 {'Class1': 0.0, 'Class2': 0.0, 'Class3': 0.0, 'Facebook': 0.0, 'Pub': 0.0, 'Pass': 0.0, 'Sleep': 0.0}
Converged after 573 iterations with delta=9.592327e-13 < theta=1.000000e-12

Final Values after Value Iteration:
      Class1: -12.543210
      Class2: 1.456790
      Class3: 4.320988
    Facebook: -22.543210
         Pub: 0.802469
        Pass: 10.000000
       Sleep: 0.000000


## Markov Decision Processes (MDP)

It introduces **actions**. The inclusion of actions means that the transition probabilities as well as the rewards must now be **conditioned on a chosen action**


<img src="../image-27.png" width="500px" /><div/>

New transition function P:

<img src="../image-62.png" width="500px" /><div/>

New reward function R:

<img src="../image-63.png" width="500px" /><div/>

There is now a separate state transition matrix and reward for every action

<img src="../image-28.png" width="500px" /><div/>

For this particular MDP, every action save for “Pub” is deterministic. In the case of the **action “ Pub”, the agent no longer enters into the “Pub” state, but instead transitions with some probability into one of the 3 “Class” states**.

Why reduce the “Pub” state to an action instead? To show that **actions can have both deterministic and stochastic effects** on where the agent ends up in the environment.

We also don’t need the “Pass” state anymore. Whereas in the MC and MRP examples we only got rewards for exiting states, we now **get rewards after taking actions**, so we can get the +10 reward for studying in Class 3 without needing to transition into a “Pass” state that immediately takes us to “Sleep”



#### Policies

Agent policy:  a function that tells the agent **what action to take depending on the state** (or a probability distribution to sample from). Policy can also be deterministic (no sampling)

Policies are **stationary** (time-independent)

<img src="../image-29.png" width="500px" /><div/>


- **deterministic**: same action is produced by policy at a given state, $a=\pi(s)$

- **stochastic**: output a probability distribution over actions again a given state (which are sampled), $\pi(a|s)=P[A_t=a|S_t=s]$

If agent follow policy $\pi$, agent probability p(a|s) of performing action a from state s equal $p(a|s)=\pi(s)$

Later: Choosing a policy impacts value function. Calculate value function under many policies and select policy with better value.


##### Backup diagrams

To capture flow of state function

<img src="../image-74.png" width="500px" /><div/>

Where:

$$P^\pi_{s,s'}=\sum_{a\in A} \pi(a|s)P^{a}_{ss'} $$

one can average MDP over all actions to create a MRP (see above Markov reward process)
So we can average any MDP to a MRP.

$$R^\pi_{s}=\sum_{a\in A} \pi(a|s)R^a_{s} $$


#### MDP Value Functions

<img src="../image-33.png" width="500px" /><div/>

**Bellman equation:**

The state-value function tells us how much reward we can expect from the current state onward, if we follow the chosen policy.

Decompose into immediate reward + discounted value of successor states


<img src="../image-35.png" width="500px" /><div/>

Expectation when we sample under the policy





<img src="../image-34.png" width="500px" /><div/>

we can use an average over many samples to estimate the action-value function

**Bellman equation**:

apply similar decomposition

<img src="../image-36.png" width="500px" /><div/>

As its stochastic, same action from same state can lead to different states -> reward received from same action and state can differ -> $q(s,a) \neq v(s')$


Bellman Expectation Equation for $V^π$

<img src="../image-37.png" width="300px" /><div/>

$$v_\pi(s)=\sum_{a\in A} \pi(a|s) q_\pi(s,a)  \tag{b.1}$$

with a stochastic policy $\pi(a|s)$ is a probability of action a given a state s

State-value function is the average of all the available action-values, weighted by how likely we are to choose them under our current policy:




-----
Bellman Expectation Equation for $Q^π$

<img src="../image-39.png" width="300px" /><div/>

$$q_\pi(s,a)=R^a_s + \gamma \sum_{s'\in S} P^a_{ss'} v_\pi(s') \tag{b.2} $$

s' is next state

if we take action a from state s, we have probability of different states

Similarly, the action-value function is the average value of the states the agent might transition to given the chosen action, but weighted instead by their transition probabilities


-----

<img src="../image-41.png" width="300px" /><div/>

we substitue $q_\pi(s,a)$ in $v_\pi(s)$ (b.1) with (b.2) in next state:

$$v_\pi(s)=\sum_{a\in A} \pi(a|s) (R^a_s + \gamma \sum_{s'\in S} P^a_{ss'} v_\pi(s')) \tag{b.3} $$



-----

<img src="../image-43.png" width="300px" /><div/>

we substitue $v_{\pi}(s')$ in $v_\pi(s)$ (b.3) with (b.1) in next state/action

$$q_\pi(s,a)=R^a_s + \gamma \sum_{s'\in S} P^a_{ss'} \sum_{a\in A} \pi(a'|s') q_\pi(s',a')  \tag{b.4} $$

s' and a' are next state/action

-----

SO we get the **value function in terms of itself**, giving us the **MDP Bellman Equation** using (b.3)

$$\pi(a|s)=0.5, \gamma=1$$

<img src="../image-45.png" width="500px" /><div/>

50% chance of going to pub or going to sleep

See in next notebook how to use Dynamic programming to compute it

## Optimal

### Optimal Value Functions

<img src="../image-47.png" width="500px" /><div/>

Optimal value function specifies the best possible performance in the MDP

**An MDP is “solved” when we know the optimal value fn.**




### Optimal policies



<img src="../image-50.png" width="500px" /><div/>

<img src="../image-51.png" width="500px" /><div/>

Optimal policy have a *.

<img src="../image-52.png" width="500px" /><div/>

means probability of a|s is 1 if $a = \text{argmax}\ q_*(s,a)$


#### Finding The Optimal Action-Value Function

Using bellman optimality equation for $v_*$

**Instead of averaging like before, we take the max value**

<img src="../image-54.png" width="500px" /><div/>




<img src="../image-56.png" width="500px" /><div/>


Then we recursively substitute

<img src="../image-58.png" width="500px" /><div/>

<img src="../image-59.png" width="500px" /><div/>


### applied to student problem

<img src="../image-77.jpg" width="500px" /><div/>


### Solving the Bellman Optimality Equation

- Bellman Optimality Equation is non-linear
- No closed form solution (in general)

- Many iterative solution methods
    - Value Iteration
    - Policy Iteration
    - Q-learning
    - Sarsa

```
The only stupid question is the one you were afraid to
ask but never did.
-Rich Sutton
```